# 09 — Existing OCR candidates to JOINED OCR output

Run the cells in order. Change `INPUT_DIR` to any directory shaped like
`TaskData/development/`; set `CANDIDATES_PATH` to its existing OCR candidates JSONL,
and choose a dedicated `OUTPUT_DIR`. This notebook joins **cached labelled sequences**
without running OCR, a GPU model, or dataset/split preparation. The current shared
`outputs/ocr/paddleocr/candidates.jsonl` covers 16 sequences / 48 development pages.
Uncached sequences are listed and skipped; they are never treated as empty OCR pages.

```text
my_directory/
  images/
    seq_example/
      01.png
      02.png
      03.png
  labels.jsonl
```

Each JSONL line has the same schema as development labels:
`{"sequence_id":"seq_example","pages":[[{"speaker":"character1","text":"HELLO!"}],[],[]]}`.
Every labelled sequence must have its own image folder and exactly three images.
Filenames are sorted lexically to define page order: use zero-padded names such as
`01.png`, `02.png`, `03.png`. Label page lists must follow that same order.
No `sequences.json` manifest or train/validation split is required.

Flow: **validate images + labels + cached candidates → optional confirmed SFX
exclusions → geometry-based line joining → JOINED OCR + label comparisons + previews**.
Labels are comparison targets only; they never replace OCR text or determine groups.
This is the deterministic joining used in the validation reports. It proposes text
regions in spatial order; it does not predict speakers or establish manga reading order.

In [ ]:
from pathlib import Path
import sys, json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif (ROOT / "manga_notebook_codebase").is_dir():
    ROOT = ROOT / "manga_notebook_codebase"
assert (ROOT / "scripts/join_directory_ocr.py").is_file(), "Run from workspace, project, or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))

# EDIT THESE: paths can also point to a directory outside TaskData.
INPUT_DIR = ROOT / "TaskData/development"  # e.g. Path(r"C:/data/my_directory")
CANDIDATES_PATH = ROOT / "outputs/ocr/paddleocr/candidates.jsonl"
OUTPUT_DIR = ROOT / "outputs/ocr_joined" / INPUT_DIR.name
LIMIT = None  # all cached labelled sequences; set 1 to preview one triplet
REVIEWS_PATH = None  # optional Path to this directory's confirmed visual SFX review JSONL

INPUT_DIR, OUTPUT_DIR = INPUT_DIR.resolve(), OUTPUT_DIR.resolve()
CANDIDATES_PATH = CANDIDATES_PATH.resolve()
print("Input:", INPUT_DIR)
print("Existing candidates:", CANDIDATES_PATH)
print("JOINED OCR output:", OUTPUT_DIR)

## 1. Validate and inspect the input

This checks duplicate IDs, image/label coverage, the three-page label schema,
and image paths before joining. The printed filenames show page alignment.
Empty label pages are allowed; they do not suppress cached OCR on those pages.
Candidates must have one record per sequence with exactly three page lists; each
fragment needs a unique page-local `id`, `text`, and `[x0, y0, x1, y1]` box.

In [ ]:
from directory_ocr import inspect_directory, load_directory_candidates

rows, references = inspect_directory(INPUT_DIR)
cached, covered = load_directory_candidates(CANDIDATES_PATH, rows)
selected = covered[:LIMIT] if LIMIT is not None else covered
uncached = [r["sequence_id"] for r in rows if r["sequence_id"] not in cached]
print(f"Found {len(rows)} labelled sequences; cached {len(covered)}; selected {len(selected)} / {3 * len(selected)} pages")
print("Uncached sequences (skipped):", uncached)
sample = selected[0]
for page_index, image_path in enumerate(sample["images"]):
    print(f"Page {page_index}: {Path(image_path).name}; {len(references[sample['sequence_id']]['pages'][page_index])} label items")
print(json.dumps(references[sample["sequence_id"]], ensure_ascii=False, indent=2))

## 2. Join the saved candidates

The notebook calls the joiner directly in its current Python kernel. It reads the
existing candidate text and boxes and generates reports; it does not initialize
PaddleOCR or invoke another interpreter. The source candidates file is read-only.
Python and Pillow are sufficient for joining and report images.

`source.json` records input image hashes and the source candidates path/hash.
Choose a new output folder when images or candidates change or when switching
directories. Reports are rebuilt for the current selection and current labels.
With `LIMIT = None`, every cached sequence matching the input labels is joined.

Without reviews, all OCR fragments are retained. To exclude visually confirmed
sound effects, supply this input's review JSONL with `sequence_id`, `page_index`,
`candidate_id`, `text`, `keep: false`, and `exclusion_category: "visual_sound_effect"`.
Stale or conflicting decisions are rejected by the existing exclusion checker.

In [ ]:
from directory_ocr import run_directory

summary = run_directory(INPUT_DIR, OUTPUT_DIR, candidates_path=CANDIDATES_PATH,
                        limit=LIMIT, reviews_path=REVIEWS_PATH)
print(json.dumps(summary, indent=2))

## 3. Read the reusable joined output

`joined.jsonl` is the main JOINED OCR artifact: one record per sequence, with three
page lists. Each group contains `id`, `text`, `box`, and `candidate_ids` linking back
to raw OCR. It contains no label-derived corrections, comparison fields, or speakers.

In [ ]:
from data_utils import load_jsonl

joined = load_jsonl(OUTPUT_DIR / "joined.jsonl")
assert len(joined) == len(selected)
assert {r["sequence_id"] for r in joined} == {r["sequence_id"] for r in selected}
assert all(len(r["pages"]) == 3 for r in joined)
print("Saved:", OUTPUT_DIR / "joined.jsonl")
print(json.dumps(joined[0], ensure_ascii=False, indent=2))

## 4. Inspect a page's text and raw/grouped images

Sample and page indices start at zero. Change these to browse the output without
rerunning OCR. `index.html` links all generated page reports; each report includes
the transcript, raw candidates, grouped boxes, and comparison with label text.

In [ ]:
from ocr_reports import get_ocr_result
from IPython.display import display
from PIL import Image

SAMPLE_INDEX = 0
PAGE_INDEX = 0
result = get_ocr_result(ROOT, sample=SAMPLE_INDEX, page=PAGE_INDEX, output=OUTPUT_DIR)
print(result["sequence_id"], "/ page", PAGE_INDEX)
print("JOINED OCR:\n" + result["transcript"])
print("Page report:", result["preview_path"])
print("All-page index:", OUTPUT_DIR / "index.html")
prefix = f"sample_{SAMPLE_INDEX}_page_{PAGE_INDEX}"
for suffix in ("raw", "groups"):
    print(suffix)
    with Image.open(OUTPUT_DIR / f"{prefix}_{suffix}.png") as image:
        preview = image.copy()
    preview.thumbnail((1000, 1400))
    display(preview)

## Saved files and interpretation

| File | Contents |
| --- | --- |
| `source.json` | Input image provenance and source candidates path/hash |
| `joined.jsonl` | Reusable joined OCR groups, independent of labels |
| `transcripts.txt` | Joined text for every selected page |
| `allocations.jsonl`, `comparison.csv` | Joined text vs label text, similarity, unmatched items |
| `summary.json` | Counts and comparison diagnostics for this selection |
| `index.html`, `pages.json` | Browse/lookup all generated page reports |
| `sample_N_page_P.*` | Per-page HTML, JSON, TXT, and raw/grouped PNGs |

Grouping links nearby vertically aligned fragments with overlapping widths and
similar font heights. Each retained candidate appears once. Comparisons use
one-to-one fuzzy text matching at similarity ≥ 0.5, ignoring punctuation and spaces;
high similarity means ≥ 0.9, not necessarily exact transcription. Unmatched label
items help locate OCR misses or grouping errors; they are not copied into the output.

For a new directory, edit the configuration cell and run all cells again. Give each
directory its own candidates file and output path, including when two input folders
share a name. The original candidates JSONL remains at `CANDIDATES_PATH`.